In [ ]:
## Middleware

"""
Middleware  provides a way to more tightly control what happens inside the agent. Middleware is useful for:

1. Traking  agent behaviour with logging, analytics and debugging
2. Transforming prompt, tool selection, and output formatting
3. Adding  retires, fallbacks, and early termination logic
4. Apply rate limit, guardrails, and PII detection """

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)


In [24]:
### Summarization of Middleware
"""for long running conversation that exceed context windows
multi tern dialouge with extensive history 
Application where preserving full conversation context matters"""

import os
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)


agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = llm,
            trigger = ("messages",10),
            keep=("messages",5)

        )
    ]
)

In [25]:
config = {"configurable": {"thread_id":"1"}}

In [ ]:
"""You ask question
      ↓
Agent checks previous conversation using thread_id
      ↓
Appends new message to old history
      ↓
Sends full conversation to model
      ↓
Stores updated conversation back in memory
      ↓
Returns full state"""


question= [
    "My name is Rahul",
    "I live in Delhi",
    "I study CS",
    "I like football",
    "I like Python",
    "What do you know about me?"
]

for q in question:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(response)
    print(len(response["messages"]))

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to inquire about various topics, including basic arithmetic operations and information about the Python programming language.\n\n## SUMMARY\nThe conversation involved answering basic arithmetic questions, such as 10+10 and 2+2, with results of 20 and 4, respectively. The user also inquired about Python, a high-level, interpreted programming language used for web development, data analysis, artificial intelligence, automation, scientific computing, and education. Python is known for its simplicity, readability, and ease of use, with key features including being easy to learn, flexible, having a large community, and being cross-platform.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone, as the conversation appears to be a series of individual inquiries without a larger task or project to complete.", additional_kwargs={'lc_source': 'summarization'}, response_meta